# CNNs for Image Classification

<a target="_blank" href="https://colab.research.google.com/github/imamitjain/notebooks/blob/main/03-deep-learning/03_cnns_image_classification.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objective:** Build convolutional neural networks for image recognition — convolution layers, pooling, architectures, and transfer learning with pretrained models.

**Prerequisites:** Neural networks (notebook 02)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q numpy matplotlib torch torchvision


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

## 1. Convolution and Pooling Operations

In [ ]:
# Visualize what convolution does
sample = torch.randn(1, 1, 28, 28)

conv = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
pool = nn.MaxPool2d(2)

conv_out = conv(sample)
pool_out = pool(conv_out)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(sample[0, 0].detach(), cmap='gray')
axes[0].set_title(f'Input {list(sample.shape)}')
axes[1].imshow(conv_out[0, 0].detach(), cmap='gray')
axes[1].set_title(f'After Conv {list(conv_out.shape)}')
axes[2].imshow(pool_out[0, 0].detach(), cmap='gray')
axes[2].set_title(f'After Pool {list(pool_out.shape)}')
plt.tight_layout()
plt.show()

print(f"Conv kernel:\n{conv.weight.data[0, 0].numpy().round(3)}")

## 2. Loading Image Data (MNIST / CIFAR-10)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# Visualize samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'Label: {label}')
    ax.axis('off')
plt.suptitle('MNIST Samples')
plt.tight_layout()
plt.show()

print(f"Training: {len(train_dataset)} images")
print(f"Test:     {len(test_dataset)} images")
print(f"Image shape: {train_dataset[0][0].shape}")

## 3. Building a CNN from Scratch

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CNN()
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Training and Evaluation Loop

In [ ]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_losses, test_accs = [], []

for epoch in range(5):
    model.train()
    running_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Evaluate
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            output = model(X_batch)
            _, predicted = output.max(1)
            total += y_batch.size(0)
            correct += predicted.eq(y_batch).sum().item()

    avg_loss = running_loss / len(train_loader)
    accuracy = correct / total
    train_losses.append(avg_loss)
    test_accs.append(accuracy)
    print(f"Epoch {epoch+1}/5 | Loss: {avg_loss:.4f} | Test Acc: {accuracy:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, 'b-o')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[1].plot(test_accs, 'r-o')
axes[1].set_title('Test Accuracy')
axes[1].set_xlabel('Epoch')
plt.tight_layout()
plt.show()

## 5. Transfer Learning with Pretrained Models

In [ ]:
# Load pretrained ResNet18 and adapt for MNIST (1-channel, 10 classes)
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze all layers
for param in resnet.parameters():
    param.requires_grad = False

# Modify first conv (3->1 channel) and last FC (1000->10 classes)
resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
resnet.fc = nn.Linear(resnet.fc.in_features, 10)

trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
total = sum(p.numel() for p in resnet.parameters())
print(f"Trainable: {trainable:,} / {total:,} parameters ({trainable/total:.1%})")
print("\nOnly conv1 and fc layers are trainable — the rest use pretrained ImageNet weights")

## 6. Visualizing Filters and Feature Maps

In [ ]:
# Visualize first-layer filters of our trained CNN
filters = model.features[0].weight.data.cpu()

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < filters.shape[0]:
        ax.imshow(filters[i, 0], cmap='gray')
    ax.axis('off')
plt.suptitle('Learned Conv1 Filters (3x3)')
plt.tight_layout()
plt.show()

# Feature maps for a single image
sample_img = test_dataset[0][0].unsqueeze(0).to(device)
with torch.no_grad():
    after_conv1 = model.features[:2](sample_img)

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < after_conv1.shape[1]:
        ax.imshow(after_conv1[0, i].cpu(), cmap='viridis')
    ax.axis('off')
plt.suptitle('Feature Maps after Conv1 + ReLU')
plt.tight_layout()
plt.show()

## Try It Yourself

1. Train a CNN on MNIST and achieve >98% test accuracy. Experiment with the number of conv layers and filters.
2. Use a pretrained ResNet18 to classify CIFAR-10. Compare accuracy when fine-tuning all layers vs. only the final classifier.
3. Visualize the first-layer filters of your trained CNN. What patterns do they detect?

In [ ]:
# Your code here